In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import RelaxedOneHotCategorical
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, entropy, ttest_ind
from sklearn.metrics import mean_absolute_error, r2_score, confusion_matrix
import gc

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def district_share_variance(district_votes: torch.Tensor, S: int) -> torch.Tensor:
    eps     = 1e-8
    #max_var = 1.0 / S
    max_var = 1.0
    totals  = district_votes.sum(dim=2, keepdim=True).clamp(min=eps)
    if (totals == 0).any():
        print("flag1!")
    share   = district_votes / totals
    mean    = share.mean(dim=2, keepdim=True)
    var     = ((share - mean) ** 2).mean(dim=2)
    return (var / max_var).clamp(0.0, 1.0)

def party_share_variance(district_votes: torch.Tensor, K: int) -> torch.Tensor:
    eps = 1e-8
    #max_var = 1.0 / K
    max_var = 1.0
    totals = district_votes.sum(dim=1, keepdim=True).clamp(min=eps)
    if (totals == 0).any():
        print("flag2!")
    shares = district_votes / totals
    mean = shares.mean(dim=1, keepdim=True)
    var = ((shares - mean) ** 2).mean(dim=1)
    return (var / max_var).clamp(0.0, 1.0)

def party_seat_share(district_votes: torch.Tensor) -> torch.Tensor:
    num_examples = district_votes.shape[0]
    num_districts = district_votes.shape[1]
    num_parties = district_votes.shape[2]
    theta = [];
    for i in range(num_examples):
        seats = np.zeros(num_parties)
        for j in range(num_districts):
            # Move the tensor to CPU and convert to NumPy array before using np.argmax
            votes = district_votes[i, j, :].detach().cpu().numpy()
            k = np.argmax(votes)
            seats[k] += 1
        theta.append(seats/num_districts)
    return np.array(theta)

In [3]:
class EncoderNN2(nn.Module):
    def __init__(self, num_districts, num_parties, hidden_dims=(256, 128, 64)):
    #def __init__(self, num_districts, num_parties, hidden_dims=(128, 64)):
        super().__init__()
        self.input_dim = num_districts * num_parties

        # Deeper network with batch normalization
        layers = []
        prev = self.input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))
            prev = h

        self.mlp = nn.Sequential(*layers)

        # Separate heads for alpha and beta
        self.alpha_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, num_parties)
        )

        self.beta1_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        self.beta2_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        self.beta3_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, phi):

        x = phi.reshape(phi.shape[0], -1) / 1
        h = self.mlp(x)
        alpha  = F.softmax(10*self.alpha_head(h),  dim=1)   # (B, K)
        beta1 = torch.sigmoid(self.beta1_head(h))
        beta2 = torch.sigmoid(self.beta2_head(h))
        beta3 = torch.sigmoid(self.beta3_head(h))
        #beta = np.stack((beta1, beta2, beta3), axis=1)
        #beta = torch.stack((beta1.squeeze(), beta2.squeeze(), beta3.squeeze()), dim=1)

        return alpha, beta1, beta2, beta3

In [4]:
import scipy

def _has_nan_grad(model):
    for p in model.parameters():
        if p.grad is not None and torch.isnan(p.grad).any():
            return True
    return False


#def train_model(data, K=3, S=50, N=5000, n_voters=500,
#                epochs=30, batch_size=32, lr=2e-4,
#                tau=1.0,
#                lambda_kl=0.5, lambda_ce=0.25):

def train_encoder_PCM(X, true_params, num_examples, num_districts, voters_per_district, num_parties,
                          device='cpu', epochs=50, batch_size=8, lr=1e-3, tau=0.5,
                          gumbel_temp=0.5, verbose=True):

    N = num_districts * voters_per_district
    K = num_parties
    S = num_districts

    encoder_model = EncoderNN2(num_districts=S, num_parties=K).to(device)
    optimizer = optim.AdamW(encoder_model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    mse_loss = nn.MSELoss()
    ce_loss = nn.CrossEntropyLoss()
    loss_history = {'total': [], 'reconstruction': [], 'variance': [], 'alpha': [], 'beta': [], 'theta': [], 'param_direct': []}
    best_loss = float('inf')

    alpha_true = torch.tensor(np.array([p[0] for p in true_params]), dtype=torch.float32, device=device)
    beta1_true = torch.tensor(np.array([p[1] for p in true_params]), dtype=torch.float32, device=device)
    beta2_true = torch.tensor(np.array([p[2] for p in true_params]), dtype=torch.float32, device=device)
    beta3_true = torch.tensor(np.array([p[3] for p in true_params]), dtype=torch.float32, device=device)

    start_time = time.time()
    indices = np.arange(num_examples)

    for ep in range(1, epochs + 1):
        encoder_model.train()
        np.random.shuffle(indices)
        epoch_loss_total = 0.0
        epoch_loss_recon = 0.0
        epoch_loss_alpha = 0.0
        epoch_loss_beta = 0.0
        epoch_loss_theta = 0.0
        epoch_loss_param = 0.0
        epoch_loss_variance = 0.0

        for i in range(0, num_examples, batch_size):
            batch_idx = indices[i:i+batch_size]
            X_range = []
            for j in batch_idx:
                x = X[j]
                distlist = np.random.choice(x.shape[0], num_districts, replace=False)
                x_sel = x[distlist]
                X_range.append(x_sel)

            phi_original = torch.tensor(np.stack(X_range), dtype=torch.float32, device=device)
            alpha_true_b = alpha_true[batch_idx]
            beta1_true_b = beta1_true[batch_idx]
            beta2_true_b = beta2_true[batch_idx]
            beta3_true_b = beta3_true[batch_idx]

            x_svar_true = district_share_variance(phi_original, num_districts)
            x_kvar_true = party_share_variance(phi_original, num_parties)
            theta_true = party_seat_share(phi_original)

            # Encoder predicts alpha, beta
            alpha_pred, beta1_pred, beta2_pred, beta3_pred = encoder_model(phi_original)

            #loss_recon += loss_variance

            # 2. Direct parameter supervision (helps with gradient flow)
            target_cls = torch.argmax(alpha_true_b, dim=1)
            #loss_alpha1_direct = ce_loss(alpha_logits, target_cls)  #predict winning party
            #loss_alpha2_direct = ce_loss(alpha_true_b, alpha_pred)
            loss_alpha3_direct = mse_loss(alpha_true_b, alpha_pred)
            loss_beta1_direct = mse_loss(beta1_true_b, beta1_pred)    #predict ABM parameter
            loss_beta2_direct = mse_loss(beta2_true_b, beta2_pred)
            loss_beta3_direct = mse_loss(beta3_true_b, beta3_pred)

            #loss_param_direct = loss_alpha1_direct + loss_alpha2_direct + loss_alpha3_direct + loss_beta_direct + loss_theta_direct
            loss = loss_beta1_direct + loss_beta2_direct + loss_beta3_direct# + loss_alpha3_direct
            loss_beta_direct = loss_beta1_direct + loss_beta2_direct + loss_beta3_direct

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(encoder_model.parameters(), max_norm=1.0)
            optimizer.step()

            # Track losses
            epoch_loss_total += loss.item() * phi_original.shape[0]
            epoch_loss_alpha += loss_alpha3_direct.item() * phi_original.shape[0]
            epoch_loss_beta += loss_beta_direct.item() * phi_original.shape[0]

          # Average losses
        epoch_loss_total /= num_examples
        epoch_loss_recon /= num_examples
        epoch_loss_variance /= num_examples
        epoch_loss_param /= num_examples
        epoch_loss_alpha /= num_examples
        epoch_loss_beta /= num_examples
        epoch_loss_theta /= num_examples

        loss_history['total'].append(epoch_loss_total)
        loss_history['reconstruction'].append(epoch_loss_recon)
        loss_history['variance'].append(epoch_loss_variance)
        loss_history['param_direct'].append(epoch_loss_param)
        loss_history['alpha'].append(epoch_loss_alpha)
        loss_history['beta'].append(epoch_loss_beta)
        loss_history['theta'].append(epoch_loss_theta)

        # Update learning rate
        scheduler.step(epoch_loss_total)

        #if verbose and (ep % 10 == 0 or ep == 1):
        if verbose :
                print(f"Epoch {ep:04d} | Total: {epoch_loss_total:.4f} | "
                  f"Recon: {epoch_loss_recon:.4f} | Param: {epoch_loss_param:.4f} | "
                  f"Variance: {epoch_loss_variance:.4f} | "
                  f"Alpha: {epoch_loss_alpha:.4f} | Beta: {epoch_loss_beta:.4f} | "
                  f"Theta: {epoch_loss_theta:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
                  f"elapsed {time.time()-start_time:.1f}s")

        # Save best model
        if epoch_loss_total < best_loss:
            best_loss = epoch_loss_total

        # Create X_tensor from the training data X (which has num_examples) for correlation calculation
        X_tensor_for_corr = torch.tensor(np.stack(X), dtype=torch.float32, device=device)
        alpha_pred_all, beta1_pred_all, beta2_pred_all, beta3_pred_all = encoder_model(X_tensor_for_corr)
        #c1 = np.correlate(alpha_pred_all.cpu().numpy(), alpha_true.cpu().numpy())
        c1 = scipy.stats.pearsonr(beta1_pred_all.detach().cpu().numpy().squeeze(), beta1_true.cpu().numpy().squeeze())[0]
        c2 = scipy.stats.pearsonr(beta2_pred_all.detach().cpu().numpy().squeeze(), beta2_true.cpu().numpy().squeeze())[0]
        c3 = scipy.stats.pearsonr(beta3_pred_all.detach().cpu().numpy().squeeze(), beta3_true.cpu().numpy().squeeze())[0]
        print("c1=", c1, "c2=", c2, "c3=", c3)

    return encoder_model, loss_history

In [5]:
import scipy
import scipy.io as sio
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
mat_file_path = '/content/drive/MyDrive/PCM_grand.mat'
data=sio.loadmat(mat_file_path)

X = data['CC']
alpha = data['alpha']
beta1 = data['beta1']
beta2 = data['beta2']
beta3 = data['beta3']
print(X.shape)
print(alpha.shape)
print(beta1.shape)
x= X[0][10]
print(x.shape)
print(alpha[500])
print(beta1[0])

NUM_DISTRICTS = X[0][0].shape[0]
NUM_PARTIES = X[0][0].shape[1]
VOTERS_PER_DISTRICT = 100 #int(np.sum(X[0][0][0]))

#voters_scaling = 100
#X = X/voters_scaling
#VOTERS_PER_DISTRICT = int(VOTERS_PER_DISTRICT/voters_scaling)
print(VOTERS_PER_DISTRICT)

train_range = range(0,500)
NUM_EXAMPLES = len(train_range)

X_np, Y, true_params = [], [], []
for i in train_range:
      X_np.append(X[0][i].astype(np.float32))
      # Fix: Removed float() conversion as beta[i] is an array, not a scalar
      Y.append((alpha[i].astype(np.float32), beta1[i].astype(np.float32), beta2[i].astype(np.float32), beta3[i].astype(np.float32)))
      true_params.append((alpha[i].astype(np.float32), beta1[i].astype(np.float32), beta2[i].astype(np.float32), beta3[i].astype(np.float32)))

Mounted at /content/drive
(1, 1000)
(1000, 3)
(1000, 1)
(100, 3)
[0.35594967 0.23224305 0.41180728]
[0.85486426]
100


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import RelaxedOneHotCategorical
import scipy.stats
import matplotlib.pyplot
import gc # Import garbage collector

from scipy.io import savemat
import numpy as np

EPOCHS = 50
BATCH_SIZE = 100  # Reduced batch size to mitigate OutOfMemoryError

#if __name__ == "__main__":
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# Clear GPU memory before starting training, if available
if device == 'cuda':
    print("Clearing GPU memory...")
    torch.cuda.empty_cache()
    gc.collect() # Force Python garbage collection
    print("GPU memory cleared.")
    print("GPU memory summary AFTER clearing cache:")
    print(torch.cuda.memory_summary())

print(f"VOTERS_PER_DISTRICT: {VOTERS_PER_DISTRICT}")
print("Training on GPU..." if device == 'cuda' else "Training on CPU...")
encoder_PCM, loss_history = train_encoder_PCM(
      X_np, true_params,
      num_examples=NUM_EXAMPLES,
      num_districts=NUM_DISTRICTS,
      num_parties=NUM_PARTIES,
      voters_per_district=VOTERS_PER_DISTRICT,
      device=device,
      epochs=EPOCHS,
      batch_size=BATCH_SIZE,
      lr=1e-3,
      gumbel_temp=0.75
    )

print("\nEncoder training complete.")

Device: cpu
VOTERS_PER_DISTRICT: 100
Training on CPU...
Epoch 0001 | Total: 0.1917 | Recon: 0.0000 | Param: 0.0000 | Variance: 0.0000 | Alpha: 0.1205 | Beta: 0.1917 | Theta: 0.0000 | LR: 0.001000 | elapsed 0.8s
c1= 0.22703534 c2= 0.15109673 c3= 0.21616308
Epoch 0002 | Total: 0.1216 | Recon: 0.0000 | Param: 0.0000 | Variance: 0.0000 | Alpha: 0.1235 | Beta: 0.1216 | Theta: 0.0000 | LR: 0.001000 | elapsed 1.5s
c1= 0.2807615 c2= 0.21202904 c3= 0.26778513
Epoch 0003 | Total: 0.0820 | Recon: 0.0000 | Param: 0.0000 | Variance: 0.0000 | Alpha: 0.1229 | Beta: 0.0820 | Theta: 0.0000 | LR: 0.001000 | elapsed 2.1s
c1= 0.28049153 c2= 0.2583304 c3= 0.31791088
Epoch 0004 | Total: 0.0617 | Recon: 0.0000 | Param: 0.0000 | Variance: 0.0000 | Alpha: 0.1290 | Beta: 0.0617 | Theta: 0.0000 | LR: 0.001000 | elapsed 2.8s
c1= 0.28367516 c2= 0.2791569 c3= 0.33743596
Epoch 0005 | Total: 0.0547 | Recon: 0.0000 | Param: 0.0000 | Variance: 0.0000 | Alpha: 0.1242 | Beta: 0.0547 | Theta: 0.0000 | LR: 0.001000 | elaps

In [7]:
Xtest = X
test_range = range(0,1000)

X_test, Y_test, test_params = [], [], []
for i in test_range:
      x = Xtest[0][i].astype(np.float32)
      X_test.append(x)
print(x.shape)

#voters_scaling = 100
#X_test = X_test/voters_scaling
X_test = np.stack(X_test)
X_test = np.ascontiguousarray(X_test)

print(X_test.shape)

with torch.no_grad():
      X_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)
      alpha_pred_all, beta1_pred_all, beta2_pred_all, beta3_pred_all = encoder_PCM(X_tensor)
      #alpha_pred = alpha_pred_all.cpu().numpy()
      #beta1_pred = beta1_pred_all.cpu().numpy()
      #beta2_pred = beta2_pred_all.cpu().numpy()
      #beta3_pred = beta3_pred_all.cpu().numpy()

beta_pred_all = torch.stack((beta1_pred_all.squeeze(), beta2_pred_all.squeeze(), beta3_pred_all.squeeze()), dim=1)

alpha_tensor = torch.tensor(alpha[test_range], dtype=torch.float32, device='cpu')
alpha_pred_all = alpha_tensor
beta_pred_all_cpu = beta_pred_all.cpu()

(100, 3)
(1000, 100, 3)


In [8]:
print(alpha_pred_all[500])
print(beta_pred_all[500])

tensor([0.3559, 0.2322, 0.4118])
tensor([0.7156, 0.7955, 0.7931])


In [9]:
from scipy.io import savemat
import numpy as np

PCM_test = {"alpha_true": alpha, "alpha_pred2": alpha_pred_all.cpu().numpy(), "beta1_true": beta1, "beta1_pred2": beta1_pred_all.cpu().numpy(), "beta2_true": beta2, "beta2_pred2": beta2_pred_all.cpu().numpy(), "beta3_true": beta3, "beta3_pred2": beta3_pred_all.cpu().numpy()}
savemat("PCM_grand_baseline.mat", PCM_test)

In [10]:
import torch.nn as nn

class EncoderNN(nn.Module):
    def __init__(self, num_districts, hidden_dims=(256, 128, 64)):
        super().__init__()
        self.input_dim = num_districts * 3

        # Deeper network with batch normalization
        layers = []
        prev = self.input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))
            prev = h

        self.mlp = nn.Sequential(*layers)

        # Separate heads for alpha and beta
        self.alpha_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 3)
        )

        self.beta_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, phi, voters_per_district):
        # Normalize input to [0, 1] range
        #x = phi.reshape(phi.shape[0], -1) / voters_per_district  # Use reshape instead of view
        x = phi.reshape(phi.shape[0], -1) / 1
        h = self.mlp(x)
        alpha_logits = self.alpha_head(h)
        alpha = torch.softmax(0.25*alpha_logits, dim=-1)
        beta = 0.5 + 0.5*torch.sigmoid(self.beta_head(h).squeeze(-1))
        return alpha, beta, alpha_logits

In [11]:
import os

#CKPT_DIR = "/content/drive/MyDrive/gradDPM_checkpoints"
#os.makedirs(CKPT_DIR, exist_ok=True)

def train_encoder_DPM(X, true_params, num_examples, num_districts, voters_per_district, num_parties,
                          device='cpu', epochs=50, batch_size=8, lr=1e-3,
                          gumbel_temp=0.5, verbose=True):
    torch.manual_seed(0)
    #X = torch.tensor(X_np, dtype=torch.float32, device=device)
    alpha_true = torch.tensor(np.array([p[0] for p in true_params]), dtype=torch.float32, device=device)
    beta_true = torch.tensor(np.array([p[1] for p in true_params]), dtype=torch.float32, device=device)
    #theta_true = torch.tensor(np.array([p[0] for p in true_params]), dtype=torch.float32, device=device)

    # num_district = num_districts.min()
    num_district = num_districts

    model = EncoderNN(num_districts=num_district).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    mse_loss = nn.MSELoss()
    ce_loss = nn.CrossEntropyLoss()

    indices = np.arange(num_examples)
    start_time = time.time()

    loss_history = {'total': [], 'reconstruction': [], 'variance': [], 'alpha': [], 'beta': [], 'theta': [], 'param_direct': []}

    best_loss = float('inf')

    for ep in range(1, epochs + 1):
        model.train()
        np.random.shuffle(indices)
        epoch_loss_total = 0.0
        epoch_loss_recon = 0.0
        epoch_loss_alpha = 0.0
        epoch_loss_beta = 0.0
        epoch_loss_theta = 0.0
        epoch_loss_param = 0.0
        epoch_loss_variance = 0.0

        indices = np.arange(num_examples)
        for i in range(0, num_examples, batch_size):
            batch_idx = indices[i:i+batch_size]
            X_range = []
            for j in batch_idx:
                x = X[j]
                X_range.append(x)
            phi_original = torch.tensor(np.stack(X_range), dtype=torch.float32, device=device)

            #phi_original = X_range
            alpha_true_b = alpha_true[batch_idx]
            beta_true_b = beta_true[batch_idx]

            # Encoder predicts alpha, beta
            alpha_pred, beta_pred, alpha_logits = model(phi_original, voters_per_district)


            #loss_recon += loss_variance

            # 2. Direct parameter supervision (helps with gradient flow)
            target_cls = torch.argmax(alpha_true_b, dim=1)
            #loss_alpha1_direct = ce_loss(alpha_logits, target_cls)  #predict winning party
            #loss_alpha2_direct = ce_loss(alpha_true_b, alpha_pred)
            loss_alpha3_direct = mse_loss(alpha_true_b, alpha_pred)
            loss_beta_direct = mse_loss(beta_pred, beta_true_b)    #predict ABM parameter
            #loss_theta_direct = mse_loss(torch.tensor(theta_reconstructed), torch.tensor(theta_true))

            #loss_param_direct = loss_alpha1_direct + loss_alpha2_direct + loss_alpha3_direct + loss_beta_direct + loss_theta_direct
            loss_param_direct = loss_alpha3_direct + loss_beta_direct #+ loss_theta_direct

            # 3. Combined loss with weighting
            # Start with more direct supervision, gradually shift to reconstruction
            #param_weight = max(0.1, 1.0 - ep / epochs)  # Decay from 1.0 to 0.1
            #recon_weight = 1.0 - param_weight + 0.1

            param_weight = 4.0
            recon_weight = 1.0
            variance_weight = 3.0

            loss = param_weight * loss_param_direct

            # Backprop
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            # Track losses
            epoch_loss_total += loss.item() * phi_original.shape[0]
            epoch_loss_param += loss_param_direct.item() * phi_original.shape[0]
            epoch_loss_alpha += loss_alpha3_direct.item() * phi_original.shape[0]
            epoch_loss_beta += loss_beta_direct.item() * phi_original.shape[0]
            #epoch_loss_theta += loss_theta_direct.item() * phi_original.shape[0]

        # Average losses
        epoch_loss_total /= num_examples
        epoch_loss_recon /= num_examples
        epoch_loss_variance /= num_examples
        epoch_loss_param /= num_examples
        epoch_loss_alpha /= num_examples
        epoch_loss_beta /= num_examples
        epoch_loss_theta /= num_examples

        loss_history['total'].append(epoch_loss_total)
        loss_history['reconstruction'].append(epoch_loss_recon)
        loss_history['variance'].append(epoch_loss_variance)
        loss_history['param_direct'].append(epoch_loss_param)
        loss_history['alpha'].append(epoch_loss_alpha)
        loss_history['beta'].append(epoch_loss_beta)
        loss_history['theta'].append(epoch_loss_theta)

        # Update learning rate
        scheduler.step(epoch_loss_total)

        #if verbose and (ep % 10 == 0 or ep == 1):
        if verbose :
            print(f"Epoch {ep:04d} | Total: {epoch_loss_total:.4f} | "
                  f"Recon: {epoch_loss_recon:.4f} | Param: {epoch_loss_param:.4f} | "
                  f"Variance: {epoch_loss_variance:.4f} | "
                  f"Alpha: {epoch_loss_alpha:.4f} | Beta: {epoch_loss_beta:.4f} | "
                  f"Theta: {epoch_loss_theta:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
                  f"elapsed {time.time()-start_time:.1f}s")

        # Save best model
        if epoch_loss_total < best_loss:
            best_loss = epoch_loss_total


    return model, loss_history

In [12]:
import scipy
import scipy.io as sio
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
mat_file_path = '/content/drive/MyDrive/DPM_grand.mat'
data=sio.loadmat(mat_file_path)

X = data['CC']
alpha = data['alpha']
beta = data['beta']
print(X.shape)
print(alpha.shape)
print(beta.shape)
x= X[0][10]
print(x.shape)

NUM_DISTRICTS = X[0][0].shape[0]
NUM_PARTIES = X[0][0].shape[1]
VOTERS_PER_DISTRICT = 100 #int(np.sum(X[0][0][0]))
print(VOTERS_PER_DISTRICT)

train_range = range(0,500)
NUM_EXAMPLES = len(train_range)

X_np, Y, true_params = [], [], []
for i in train_range:
      X_np.append(X[0][i].astype(np.float32))
      Y.append((alpha[i].astype(np.float32), beta[0][i].astype(np.float32)))
      true_params.append((alpha[i].astype(np.float32), beta[0][i].astype(np.float32)))

Mounted at /content/drive
(1, 1000)
(1000, 3)
(1, 1000)
(100, 3)
100


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import RelaxedOneHotCategorical
import scipy.stats
import matplotlib.pyplot
import gc # Import garbage collector

from scipy.io import savemat
import numpy as np

EPOCHS = 50
BATCH_SIZE = 100  # Reduced batch size to mitigate OutOfMemoryError

#if __name__ == "__main__":
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# Clear GPU memory before starting training, if available
if device == 'cuda':
    print("Clearing GPU memory...")
    torch.cuda.empty_cache()
    gc.collect() # Force Python garbage collection
    print("GPU memory cleared.")
    print("GPU memory summary AFTER clearing cache:")
    print(torch.cuda.memory_summary())

print(f"VOTERS_PER_DISTRICT: {VOTERS_PER_DISTRICT}")
print("Training on GPU..." if device == 'cuda' else "Training on CPU...")
encoder_DPM, loss_history = train_encoder_DPM(
      X_np, true_params,
      num_examples=NUM_EXAMPLES,
      num_districts=NUM_DISTRICTS,
      num_parties=NUM_PARTIES,
      voters_per_district=VOTERS_PER_DISTRICT,
      device=device,
      epochs=EPOCHS,
      batch_size=BATCH_SIZE,
      lr=1e-3,
      gumbel_temp=0.75
    )

print("\nEncoder training complete.")

Device: cpu
VOTERS_PER_DISTRICT: 100
Training on CPU...
Epoch 0001 | Total: 0.1961 | Recon: 0.0000 | Param: 0.0490 | Variance: 0.0000 | Alpha: 0.0288 | Beta: 0.0202 | Theta: 0.0000 | LR: 0.001000 | elapsed 0.1s
Epoch 0002 | Total: 0.1564 | Recon: 0.0000 | Param: 0.0391 | Variance: 0.0000 | Alpha: 0.0242 | Beta: 0.0150 | Theta: 0.0000 | LR: 0.001000 | elapsed 0.1s
Epoch 0003 | Total: 0.1283 | Recon: 0.0000 | Param: 0.0321 | Variance: 0.0000 | Alpha: 0.0207 | Beta: 0.0114 | Theta: 0.0000 | LR: 0.001000 | elapsed 0.2s
Epoch 0004 | Total: 0.1059 | Recon: 0.0000 | Param: 0.0265 | Variance: 0.0000 | Alpha: 0.0176 | Beta: 0.0089 | Theta: 0.0000 | LR: 0.001000 | elapsed 0.2s
Epoch 0005 | Total: 0.0852 | Recon: 0.0000 | Param: 0.0213 | Variance: 0.0000 | Alpha: 0.0144 | Beta: 0.0069 | Theta: 0.0000 | LR: 0.001000 | elapsed 0.3s
Epoch 0006 | Total: 0.0677 | Recon: 0.0000 | Param: 0.0169 | Variance: 0.0000 | Alpha: 0.0114 | Beta: 0.0056 | Theta: 0.0000 | LR: 0.001000 | elapsed 0.3s
Epoch 0007 | T

In [14]:
Xtest_raw = data['CC']

X_test_processed = []
for i in range(Xtest_raw[0].shape[0]): # Iterate through the actual number of samples in Xtest_raw[0]
    x = Xtest_raw[0][i].astype(np.float32)
    X_test_processed.append(x)

X_tensor = torch.tensor(np.stack(X_test_processed), dtype=torch.float32, device=device)
alpha_pred, beta_pred, _ = encoder_DPM(X_tensor, VOTERS_PER_DISTRICT)
alpha_pred = alpha_pred.detach().cpu().numpy()
beta_pred = beta_pred.detach().cpu().numpy()

print(alpha_pred[500])
print(beta_pred[500])

[0.49478322 0.2368122  0.2684045 ]
0.9111336


In [15]:
from scipy.io import savemat
import numpy as np
DPM_test = {"alpha_true": alpha, "alpha_pred2": alpha_pred, "beta_true": beta, "beta_pred2": beta_pred}
savemat("DPM_grand_baseline.mat", DPM_test)